[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 1: The Pen & Paper Method (Markov Decision Process)

### Key assumptions
- We have a finite time horizon with known demand patterns
- Staffing decisions in one period affect the next period's state
- Labor costs, understaffing penalties, and overstaffing penalties are known
- We want to minimize total costs while meeting service requirements

### Approach (step-by-step)
1. **Define the MDP components**: States, actions, transitions, rewards
2. **Formulate the reward function**: Include labor, understaffing, and overstaffing costs
3. **Apply Value Iteration**: Find the optimal policy using dynamic programming
4. **Extract the optimal schedule**: From the converged value function and policy

### What to look for in the results
- Optimal staffing levels for each time period
- Total minimum cost over the planning horizon
- Policy that balances labor costs with service penalties
- Trade-offs between overstaffing and understaffing

### Concrete example (from the source)
3-hour period with varying demand:
- Time Periods: t ∈ {1, 2, 3}
- Demand (required staff): D = {d₁=1, d₂=3, d₃=2}
- Staffing levels: n ∈ {0, 1, 2, 3}
- Costs: Labor = $20/hr, Understaffing = $50/hr, Overstaffing = $10/hr

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Define the MDP components for workforce scheduling
class WorkforceMDP:
    """Markov Decision Process for workforce scheduling"""
    
    def __init__(self, time_periods: List[int], demand: List[int], 
                 max_staff: int, labor_cost: float, 
                 understaff_penalty: float, overstaff_penalty: float,
                 gamma: float = 0.95):
        """
        Initialize the Workforce MDP
        
        Args:
            time_periods: List of time period indices
            demand: Required staff for each time period
            max_staff: Maximum number of staff that can be scheduled
            labor_cost: Cost per staff member per hour
            understaff_penalty: Penalty per missing staff member
            overstaff_penalty: Penalty per extra staff member
            gamma: Discount factor for future rewards
        """
        self.time_periods = time_periods
        self.demand = demand
        self.max_staff = max_staff
        self.labor_cost = labor_cost
        self.understaff_penalty = understaff_penalty
        self.overstaff_penalty = overstaff_penalty
        self.gamma = gamma
        
        # Create state space: (time_period, current_staff)
        self.states = [(t, n) for t in time_periods for n in range(max_staff + 1)]
        
        # Create action space: change in staffing level
        self.actions = list(range(-max_staff, max_staff + 1))
        
        # Initialize value function and policy
        self.values = {state: 0.0 for state in self.states}
        self.policy = {state: 0 for state in self.states}
    
    def calculate_reward(self, state: Tuple[int, int], action: int) -> float:
        """
        Calculate the immediate reward for a state-action pair
        
        Args:
            state: Current state (time_period, current_staff)
            action: Change in staffing level
        
        Returns:
            Immediate reward (negative cost)
        """
        t, current_staff = state
        
        # Calculate new staffing level
        new_staff = max(0, min(self.max_staff, current_staff + action))
        
        # Get demand for current time period
        demand_t = self.demand[t] if t < len(self.demand) else 0
        
        # Calculate costs
        labor_cost = new_staff * self.labor_cost
        
        # Understaffing cost (penalty for not meeting demand)
        if new_staff < demand_t:
            understaff_cost = (demand_t - new_staff) * self.understaff_penalty
        else:
            understaff_cost = 0
        
        # Overstaffing cost (penalty for having too many staff)
        if new_staff > demand_t:
            overstaff_cost = (new_staff - demand_t) * self.overstaff_penalty
        else:
            overstaff_cost = 0
        
        # Total cost (negative reward)
        total_cost = labor_cost + understaff_cost + overstaff_cost
        
        return -total_cost
    
    def get_next_state(self, state: Tuple[int, int], action: int) -> Tuple[int, int]:
        """
        Get the next state given current state and action
        
        Args:
            state: Current state (time_period, current_staff)
            action: Change in staffing level
        
        Returns:
            Next state (next_time_period, new_staff)
        """
        t, current_staff = state
        
        # Calculate new staffing level
        new_staff = max(0, min(self.max_staff, current_staff + action))
        
        # Move to next time period
        next_t = t + 1
        
        # If we've reached the end of the planning horizon, stay in terminal state
        if next_t >= len(self.time_periods):
            next_t = len(self.time_periods) - 1
        
        return (next_t, new_staff)
    
    def value_iteration(self, max_iterations: int = 1000, tolerance: float = 1e-6) -> None:
        """
        Perform value iteration to find the optimal policy
        
        Args:
            max_iterations: Maximum number of iterations
            tolerance: Convergence tolerance
        """
        for iteration in range(max_iterations):
            delta = 0
            new_values = {}
            
            for state in self.states:
                t, current_staff = state
                
                # Skip terminal states
                if t >= len(self.time_periods) - 1:
                    new_values[state] = 0
                    continue
                
                # Find the best action
                best_value = float('-inf')
                best_action = 0
                
                for action in self.actions:
                    # Calculate immediate reward
                    reward = self.calculate_reward(state, action)
                    
                    # Get next state
                    next_state = self.get_next_state(state, action)
                    
                    # Calculate expected value
                    expected_value = reward + self.gamma * self.values[next_state]
                    
                    # Update best value and action
                    if expected_value > best_value:
                        best_value = expected_value
                        best_action = action
                
                new_values[state] = best_value
                self.policy[state] = best_action
                
                # Track convergence
                delta = max(delta, abs(new_values[state] - self.values[state]))
            
            # Update values
            self.values = new_values
            
            # Check for convergence
            if delta < tolerance:
                print(f"Value iteration converged after {iteration + 1} iterations")
                break
    
    def extract_optimal_schedule(self, initial_staff: int) -> Dict:
        """
        Extract the optimal schedule from the learned policy
        
        Args:
            initial_staff: Initial number of staff
        
        Returns:
            Dictionary containing the optimal schedule and costs
        """
        schedule = []
        total_cost = 0
        
        # Start from initial state
        current_state = (0, initial_staff)
        
        for t in range(len(self.time_periods)):
            if t >= len(self.time_periods) - 1:
                break
            
            # Get optimal action
            action = self.policy[current_state]
            
            # Calculate new staffing level
            new_staff = max(0, min(self.max_staff, current_state[1] + action))
            
            # Calculate costs for this period
            demand_t = self.demand[t]
            labor_cost = new_staff * self.labor_cost
            
            if new_staff < demand_t:
                understaff_cost = (demand_t - new_staff) * self.understaff_penalty
            else:
                understaff_cost = 0
            
            if new_staff > demand_t:
                overstaff_cost = (new_staff - demand_t) * self.overstaff_penalty
            else:
                overstaff_cost = 0
            
            period_cost = labor_cost + understaff_cost + overstaff_cost
            total_cost += period_cost
            
            # Add to schedule
            schedule.append({
                'period': t + 1,
                'demand': demand_t,
                'staff': new_staff,
                'action': action,
                'labor_cost': labor_cost,
                'understaff_cost': understaff_cost,
                'overstaff_cost': overstaff_cost,
                'total_cost': period_cost
            })
            
            # Move to next state
            current_state = self.get_next_state(current_state, action)
        
        return {
            'schedule': schedule,
            'total_cost': total_cost,
            'initial_staff': initial_staff
        }

In [ ]:
# Create the MDP instance with the concrete example from the source
time_periods = [0, 1, 2]  # 3 time periods
demand = [1, 3, 2]  # Required staff for each period
max_staff = 3  # Maximum staff that can be scheduled
labor_cost = 20  # $20 per hour per staff
understaff_penalty = 50  # $50 per hour per missing staff
overstaff_penalty = 10  # $10 per hour per extra staff

# Create and solve the MDP
mdp = WorkforceMDP(time_periods, demand, max_staff, labor_cost, 
                   understaff_penalty, overstaff_penalty)

print("MDP Components:")
print(f"Time periods: {time_periods}")
print(f"Demand: {demand}")
print(f"Max staff: {max_staff}")
print(f"Labor cost: ${labor_cost}/hr")
print(f"Understaff penalty: ${understaff_penalty}/hr")
print(f"Overstaff penalty: ${overstaff_penalty}/hr")
print(f"Number of states: {len(mdp.states)}")
print(f"Number of actions: {len(mdp.actions)}")

MDP Components:
Time periods: [0, 1, 2]
Demand: [1, 3, 2]
Max staff: 3
Labor cost: $20/hr
Understaff penalty: $50/hr
Overstaff penalty: $10/hr
Number of states: 12
Number of actions: 7


In [ ]:
# Run value iteration to find the optimal policy
mdp.value_iteration()

print("\nValue iteration completed!")
print(f"Discount factor (gamma): {mdp.gamma}")

Value iteration converged after 3 iterations

Value iteration completed!
Discount factor (gamma): 0.95


In [ ]:
# Extract the optimal schedule starting with 1 staff member
result = mdp.extract_optimal_schedule(initial_staff=1)

# Display the optimal schedule
schedule_df = pd.DataFrame(result['schedule'])
print("Optimal Workforce Schedule:")
print(schedule_df.to_string(index=False))

print(f"\nTotal Cost: ${result['total_cost']:.2f}")
print(f"Initial Staff: {result['initial_staff']}")

Optimal Workforce Schedule:
 period  demand  staff  action  labor_cost  understaff_cost  overstaff_cost  total_cost
      1       1      1       0          20                0               0          20
      2       3      3       2          60                0               0          60

Total Cost: $80.00
Initial Staff: 1


In [ ]:
try:
    # Create comprehensive visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Workforce Scheduling MDP Analysis', fontsize=16, fontweight='bold')
    
    # 1. Demand vs Staff Schedule
    ax1 = axes[0, 0]
    periods = schedule_df['period']
    demand_vals = schedule_df['demand']
    staff_vals = schedule_df['staff']
    
    ax1.plot(periods, demand_vals, 'ro-', label='Demand', linewidth=2, markersize=8)
    ax1.plot(periods, staff_vals, 'bs-', label='Scheduled Staff', linewidth=2, markersize=8)
    ax1.set_xlabel('Time Period')
    ax1.set_ylabel('Number of Staff')
    ax1.set_title('Demand vs Optimal Staff Schedule')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(periods)
    
    # 2. Cost Breakdown
    ax2 = axes[0, 1]
    cost_breakdown = schedule_df[['labor_cost', 'understaff_cost', 'overstaff_cost']]
    bottom = np.zeros(len(schedule_df))
    colors = ['#2ecc71', '#e74c3c', '#f39c12']
    labels = ['Labor Cost', 'Understaff Cost', 'Overstaff Cost']
    
    for i, (col, color, label) in enumerate(zip(cost_breakdown.columns, colors, labels)):
        ax2.bar(periods, cost_breakdown[col], bottom=bottom, 
                color=color, alpha=0.8, label=label)
        bottom += cost_breakdown[col]
    
    ax2.set_xlabel('Time Period')
    ax2.set_ylabel('Cost ($)')
    ax2.set_title('Cost Breakdown by Period')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(periods)
    
    # 3. Total Cost per Period
    ax3 = axes[1, 0]
    ax3.bar(periods, schedule_df['total_cost'], color='#3498db', alpha=0.8)
    ax3.set_xlabel('Time Period')
    ax3.set_ylabel('Total Cost ($)')
    ax3.set_title('Total Cost per Period')
    ax3.grid(True, alpha=0.3)
    ax3.set_xticks(periods)
    
    # Add value labels on bars
    for i, cost in enumerate(schedule_df['total_cost']):
        ax3.text(periods[i], cost + 5, f'${cost}', ha='center', va='bottom')
    
    # 4. Staffing Actions
    ax4 = axes[1, 1]
    actions = schedule_df['action']
    colors_action = ['green' if a >= 0 else 'red' for a in actions]
    ax4.bar(periods, actions, color=colors_action, alpha=0.7)
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax4.set_xlabel('Time Period')
    ax4.set_ylabel('Staffing Action')
    ax4.set_title('Staffing Changes (Positive=Add, Negative=Remove)')
    ax4.grid(True, alpha=0.3)
    ax4.set_xticks(periods)
    
    # Add value labels on bars
    for i, action in enumerate(actions):
        ax4.text(periods[i], action + 0.1 if action >= 0 else action - 0.1, 
                 f'{action:+d}', ha='center', va='bottom' if action >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
# Calculate reward for the specific example from the source
# State transition: t=2 with n=3 staff, action to reduce to n'=2 for t=3
state = (1, 3)  # t=2 (0-indexed), n=3 staff
action = -1     # Reduce staff by 1

reward = mdp.calculate_reward(state, action)
print(f"Reward for state transition (t=2, n=3) -> (t=3, n=2): ${reward:.2f}")

# Manual calculation verification
print("\nManual calculation:")
print(f"Labor cost: 2 staff × ${labor_cost} = ${2 * labor_cost}")
print(f"Understaffing: Need 3, have 2 → {(3-2)} × ${understaff_penalty} = ${(3-2) * understaff_penalty}")
print(f"Overstaffing: 0")
print(f"Total cost: ${2 * labor_cost + (3-2) * understaff_penalty}")
print(f"Reward (negative cost): -${2 * labor_cost + (3-2) * understaff_penalty}")

Reward for state transition (t=2, n=3) -> (t=3, n=2): $-90.00

Manual calculation:
Labor cost: 2 staff × $20 = $40
Understaffing: Need 3, have 2 → 1 × $50 = $50
Overstaffing: 0
Total cost: $90
Reward (negative cost): -$90


In [ ]:
# Sensitivity analysis: What if we start with different initial staff?
initial_staff_levels = [0, 1, 2, 3]
results = []

for initial_staff in initial_staff_levels:
    result = mdp.extract_optimal_schedule(initial_staff)
    results.append({
        'initial_staff': initial_staff,
        'total_cost': result['total_cost'],
        'final_staff': result['schedule'][-1]['staff'] if result['schedule'] else 0
    })

sensitivity_df = pd.DataFrame(results)
print("Sensitivity Analysis - Impact of Initial Staff Level:")
print(sensitivity_df.to_string(index=False))

# Visualize sensitivity
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.bar(sensitivity_df['initial_staff'], sensitivity_df['total_cost'], 
        color='#3498db', alpha=0.8)
plt.xlabel('Initial Staff Level')
plt.ylabel('Total Cost ($)')
plt.title('Total Cost vs Initial Staff Level')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(sensitivity_df['initial_staff'], sensitivity_df['total_cost'], 
         'bo-', linewidth=2, markersize=8)
plt.xlabel('Initial Staff Level')
plt.ylabel('Total Cost ($)')
plt.title('Cost Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Sensitivity Analysis - Impact of Initial Staff Level:
 initial_staff  total_cost  final_staff
             0          80            3
             1          80            3
             2          80            3
             3          80            3


### Why this Tier exists vs earlier Tiers
This is Tier 1, so it establishes the mathematical foundation for workforce scheduling. The MDP approach:
- Provides **exact optimality** through dynamic programming
- Captures the **sequential decision-making** nature of scheduling
- Offers a **systematic framework** for understanding the problem structure
- Serves as a **benchmark** for evaluating heuristic and metaheuristic approaches

### Pros / Cons vs other approaches
**Pros:**
- ✅ Guaranteed optimal solution for the given problem formulation
- ✅ Transparent decision-making process
- ✅ Handles uncertainty through probabilistic transitions
- ✅ Provides theoretical foundation for understanding the problem

**Cons:**
- ❌ Computationally expensive for large problems (curse of dimensionality)
- ❌ Requires complete knowledge of demand and costs
- ❌ May not scale well for many time periods or staff members
- ❌ Assumes deterministic transitions (simplified from real-world uncertainty)

### When to use this Tier
- **Small to medium-sized problems** with limited time horizons
- **Strategic planning** where optimality is crucial
- **Benchmarking** other algorithms against the optimal solution
- **Academic settings** where theoretical understanding is important
- **High-value decisions** where the computational cost is justified